# 07 · Диагностика: где и в какой форме LoRA меняет модель

**Зачем** (JOURNAL, 2026-10-27). LoRA на тех же данных закрывает разрыв почти полностью — это маркер:
нужное поведение достижимо малым изменением. Цель следующего этапа — без градиентного обучения привести
модель туда же. Прежде чем строить методы, выясняем:

1. **Где живёт эффект.** Подставляем скрытые состояния модели с LoRA в базовую — на одном слое и в одном
   наборе позиций за раз — и смотрим, становится ли решение как у LoRA (0 — как у базы, 1 — как у LoRA).
2. **Какой он формы.** Разница активаций Δh = h_LoRA − h_база: какую её долю на новых задачах
   воспроизводит постоянный вектор (класс CAA), вектор по состоянию «подходит / не подходит» и линейное
   отображение состояния базы. Плюс поведенческая проверка: добавка каждой формы в точке решения.

Данные — train (на них учились вектор и LoRA). Ничего не подбирается; val и test не используются.
Результат разведочный.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # Mac: неподдержанные операции -> CPU

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

from steering_tool_feedback.config import ExperimentConfig
from steering_tool_feedback.pipeline import Experiment
from steering_tool_feedback import journal, viz

In [ ]:
cfg = ExperimentConfig.make(
    name="lora-diag-qwen3-1.7b", hypothesis="explore", notebook="notebooks/07_lora_diagnostics.ipynb",
    notes="Где и в какой форме LoRA меняет активации: подстановка LoRA → база, разбор Δh.",
    overrides={
        "model.name": "Qwen/Qwen3-1.7B",
        "data.n_tasks": 160,
        "data.generator": "v3",
        "eval.methods": ["baseline", "strong_prompt", "restate", "explicit_state",
                         "steering", "steering_gated", "tact", "lora"],
        "steer.alphas": [0.25, 0.5, 1.0, 2.0, 4.0],
    },
)
exp = Experiment(cfg)
exp.run("fit")

### Диагностика

In [ ]:
diag = exp.lora_diagnostics()
{k: diag[k] for k in ("n_prompts", "n_disagree", "n_control", "lora_fixes", "lora_breaks",
                      "first_token_agreement", "adapter")}

### 1. Где живёт эффект LoRA: подстановка по слоям и позициям

In [ ]:
summ = pd.DataFrame(diag["patch_summary"])
viz.patch_heatmap(summ);
summ.pivot(index="set", columns="layer", values="effect").round(2)

### 2. Какой формы изменение, которое делает LoRA

In [ ]:
dd = pd.DataFrame(diag["decomposition"])
for pos in ("last", "decision", "tool_content"):
    viz.delta_forms(dd, pos)
dd[dd.position == "last"][["layer", "r2_const", "r2_probe", "r2_state", "r2_linear", "rank90_centered", "norm_mean"]].round(3)

### Поведенческая проверка формы (CV по задачам train)

Добавка в точку решения на лучших по подстановке слоях. `fixed_like_lora` — доля расхождений, где решение стало как у LoRA; `kept_where_agree` — доля совпадений, где решение не сломалось.

In [ ]:
pd.DataFrame(diag["form_check"]).round(3)

## Вывод → журнал

In [ ]:
display(Markdown(exp.report(
    conclusion="""…""",
    next_step="""…""",
)))